# Machine Learning

Herein we will be introducing and discussing some machine learning models at a high level to get first time and beginner biomedical informaticians a taste for some of the problems machine learning can help solve. Furthermore, we will implement both a logisitic regression model and a deep neural network (DNN) and evaluate the efficacy of these models on an example dataset.

As this is a brief survey course on machine learning, we will not dive too deep into the theory of these models, nor will we be able to cover every type of model. At the end of this notebook I have provided a list of references and resources that biomedical informaticians, beginner, novice, and experts alike, may find useful and insightful.

<img width="800px" src="img/extended_ml_cheat_sheet.jpeg"/>

[An Extended Version Of The Scikit-Learn Cheat Sheet](https://medium.com/@chris_bour/an-extended-version-of-the-scikit-learn-cheat-sheet-5f46efc6cbb)

**<font color='red'>Disclaimer</font>:** I will not be doing any data cleaning (per se) or data manipulation. This is a very large topic of its own that I will not have time to cover. However, understanding the dataset with which you are working, ensuring it is clean, workable data, and formulating hypotheses you wish to test with your data are all ***VERY*** important aspects to consider before diving into machine or deep learning models with your data. **The above figure, and corresponding article, depict this point excellently.**

*I aim to provide rudimentary information about machine learning that will help you start to understand different models and when to use them. I hope to garner excitement about this field so that you go and learn more about machine learning and how to apply it to your own unique biomedical informatics problems!*

https://scikit-learn.org/stable/common_pitfalls.html

---

## Types of machine learning

There are many different types of models in machine learning and choosing the best one is dependent on:
1. The problem you aim to solve
2. The data you have

In some instances multiple models may work well for you, in which case you will have to consider other aspects of the model, such as:
* interpretability
* memory cost
* number of samples
* dimensionality
* and so on...

Though these considerations may help you narrow down your choices, choosing the *best* remains a difficult task. I will provide some general information about different types of machine learning models while keeping some of the above aspects in mind.

Below is a figure that shows a very well defined hierarchy of different ML models that one can consider. The upper level of this hierarchy gives 3 main learning types: **supervised**, **unsupervised**, and **reinforcement**. I will discuss all 3 of these as well as a fourth, called **semi-supervised**.

<img width="500px" src="img/ml_hierarchy.png"/>


### Supervised learning

Samples of input-output pairs (labelled outcomes)

**Classification** - predict the binary (or class) label for an unlabeled sample. Examples: logistic regression, SVM

**Regression** - predict a real-valued label for an unlabeled sample. Examples: linear regression 

<img width="400px" src="img/class_v_reg.png"/>

In classification models, the boundary separating the examples of different classes is called the *decision boundary*. For regression models, the line that best fits the data is the *regression line*. 

https://en.wikipedia.org/wiki/Supervised_learning

https://scikit-learn.org/stable/supervised_learning.html

*Logistic regression and the vanilla neural network we will implement herein are both considered supervised classification models. Therefore, this model type will be our main focus for this notebook.*

### Unsupervised learning

Draw inferences and patterns from input data without labelled output data. Examples: k-means, PCA

<img width="400px" src="img/clusters.png"/>

https://en.wikipedia.org/wiki/Unsupervised_learning

https://scikit-learn.org/stable/unsupervised_learning.html

### Semi-supervised learning

Supervised learning tasks and techniques that also make use of unlabeled data for training.

https://en.wikipedia.org/wiki/Semi-supervised_learning

https://scikit-learn.org/stable/modules/classes.html#module-sklearn.semi_supervised

### Reinforcement learning

Model tries a bunch of different things and is given a reward signal when doing something well.

https://en.wikipedia.org/wiki/Reinforcement_learning


## Logistic regression
Logistic regression is a simple and commonly used machine learning algorithm for two-class classification. It is used to describe data and to explain the relationship between one dependent binary variable and one or more nominal, ordinal, interval or ratio-level independent variables. Logistic regression can be used to answer questions such as:
* How does the probability of getting lung cancer (yes vs. no) change for every additional pound a person is overweight and for every pack of cigarettes smoked per day?
* Do body weight, calorie intake, fat intake, and age have an influence on the probability of having a heart attack (yes vs. no)?

*Logistic regression can also be used for multi-class predictions, but we will not cover that here.*

In general, logistic regression uses a linear combination of more than one feature value or explanatory variable as argument of the sigmoid function:

$f(x) = \frac{1}{1+e^{-x}}$

The corresponding output of the sigmoid function is a number between 0 and 1. 

<img width="300px" src="img/sigmoid.png"/>

The middle value is considered as threshold to establish what belongs to the class 1 and to the class 0. In particular, an input producing an outcome greater than 0.5 is considered belonging to the class 1. Conversely, if the output is less than 0.5, then the corresponding input is classified as belonging to class 0.

For our logistic regression model we use the logistic function:

$f_{w,b}(x) = \frac{1}{1+e^{-(wx+b)}}$

The logistic function is our **activation function**. This is going to tell us when a sample is 0 or 1.

To calculate the solution to this equation, i.e. obtain the best intercept and coefficients, we aim to maximize the **log likelihood** of the training data.

$ \ln L_{\mathbf{w},b} = \sum_{i=1}^{N}y_{i}\ln f_{\mathbf{w},b}(x_{i})+(1-{y_{i}}) \ln (1-f_{\textbf{w},b}(x_{i})$

Though it may appear daunting, when you break it down, it isn't that bad. When $y_{i}=1$, the second part of the summation drops out (1-1=0), whereas when $y_{i}=0$ the first part of the equation drops out. 

Log likelihood is our **cost function**. Generally, minimizing functions is preferred over maximizing, so the negative of the function is commonly used.

## Optimizers

The general point is there are many methods that have been developed and well tested for optimizing (maximizing or minimizing) a function. You start with a guess then you adjust the parameters over several iterations until you have converged to some point (number of iterations, tolerance, etc.).

<img width="300px" src="img/minimization.gif"/>


I do not go into specific optimizers here, for the sake of time. I do think I will update this notebook in the future to include a nice section on optimizers.

https://en.wikipedia.org/wiki/Gradient_descent

https://scikit-learn.org/stable/modules/sgd.html

https://en.wikipedia.org/wiki/Limited-memory_BFGS

For our implementation of logistic regression, we will use scikit-learn's LogisticRegression model:
* https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

Let's load the libraries we will be using...

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, \
    f1_score, roc_auc_score, auc, precision_recall_curve, roc_curve,\
    classification_report, confusion_matrix

## Pima Indians Diabetes dataset
We will use the Pima Indians dataset to experiment with decision trees. The Pima are a group of Native Americans living in Arizona. A genetic predisposition allowed this group to survive on a carbohydrate poor diet. In recent years, a sudden shift from traditional agricultural crops to processed foods and a decline in physical activity led to a high prevalence of type 2 diabetes in this population.  

The dataset can be downloaded here:

https://www.kaggle.com/uciml/pima-indians-diabetes-database#diabetes.csv

I have named the downloaded file: `diabetes.csv`.

The dataset includes data from 768 women. The columns are defined as follows:

* `Pregnancies`: Number of times pregnant
* `Glucose`: Plasma glucose concentration a 2 hours in an oral glucose tolerance test
* `BloodPressure`: Diastolic blood pressure (mm Hg)
* `SkinThickness`: Triceps skin fold thickness (mm)
* `Insulin`: 2-Hour serum insulin (mu U/ml)
* `BMI`: Body mass index (weight in kg/(height in m)^2)
* `DiabetesPedigreeFunction`: The output of the pedigree function that provides measure of genetic influence and gives us an idea of the hereditary risk one might have with the onset of diabetes mellitus
* `Age`: Age (years)
* `Outcome`: Class variable (0 or 1) 268 of 768 are 1 (positive), the others are 0 (negative)

In [ ]:
## load Pima Indians Diabetes dataset (downloaded May 14, 2019; N=768)
df = pd.read_csv("diabetes.csv")

## Missingness

The only cleaning we need to do is to drop the rows that contain missing values. In general practice, you do not remove these rows without further exploratory analysis. However, for sake of this example, we have omitted rows that contain missing values.

In [ ]:
## function to determine if a row has an missing value
def valid_value(row):
    if 0 == row['Glucose'] or \
       0 == row['BloodPressure'] or \
       0 == row['SkinThickness'] or \
       0 == row['Insulin'] or \
       0 == row['BMI'] or \
       0 == row['Age']:
        return False
    else:
        return True

## create dataframe with only valid rows
df_pima = df[df.apply(lambda row: valid_value(row), axis=1)]
df_pima.head()

In [ ]:
print(f"length of original dataframe: {len(df)}")
print(f"length of filtered dataframe: {len(df_pima)}")

## Splitting samples

Now we split the data into our training and test sets. We do this because we need to train the model on some of the data and ensure that we have generalizable model by testing the optimized model on samples it has never seen.

<img width="600px" src="img/data.png" />

First, we separate the features. By convention, scikit-learn often refers to the feature dataset as `X` and the target dataset as `y`.


In [ ]:
## split dataset in features and target variable
feature_cols = \
    ['Pregnancies', 'Insulin', 'BMI', 'Age','Glucose',
     'BloodPressure','DiabetesPedigreeFunction', 'SkinThickness']

X = df_pima[feature_cols]
y = df_pima['Outcome']

In [ ]:
## split dataset into training set and test set
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.3, random_state=42) # 70% training and 30% test
print(len(X_train))
print(len(X_test))

Finally, the test and training data is fit to our model and we predict outcomes.

In [ ]:
## create a logistic regression classifier and predict
model = LogisticRegression(random_state=42, max_iter=500)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
print(f"Our model converged after {model.n_iter_[0]} iterations.")

## Evaluating the model
To evaluate our logistic regression model, we will examine the confusion matrix, accuracy score, precision score, recall score, and f1 score. 

**Accuracy:**
* $\frac{TP + TN}{TP + TN + FP + FN}$
* Accuracy is the ratio of correct predictions to total predictions made. However, there are problems with accuracy. It assumes equal costs for both kinds of errors. A 99% accuracy can be excellent, good, mediocre, poor or terrible depending upon the problem.

**Precision:**
* $\frac{TP}{TP + FP}$
* Precision is the ability of a classifier not to label an instance positive that is actually negative. 
* High precision indicates a small number of false positives.

**Recall:**
* $\frac{TP}{TP + FN}$
* Recall is the ability of a classifier to find all positive instances. 
* High recall indicates a small number of false negatives.

<img width="300px" src="img/precision-recall.png" />

**F1 score (F measure):**
* $\frac{2 * Recall * Precision}{Recall + Precision}$
* Since we have two measures (Precision and Recall) it helps to have a measurement that represents both of them. We calculate an F1 score that uses Harmonic Mean in place of Arithmetic Mean as it punishes the extreme values more. 

In [ ]:
def show_confusion_matrix(y_test, y_pred, palette="Blues"):
    ## see: https://www.geeksforgeeks.org/confusion-matrix-machine-learning/
    ##      https://jakevdp.github.io/PythonDataScienceHandbook/05.08-random-forests.html
    ##      https://classeval.wordpress.com/introduction/basic-evaluation-measures/
    matrix = confusion_matrix(y_test, y_pred)

    colors = sns.color_palette(palette) # set the colors to use for heatmap

    ax = sns.heatmap(matrix, square=True, annot=True, fmt='d', 
                     cbar=False, cmap=colors, vmin=-1, annot_kws={"size":13}, linewidths=1.0)

    # set labels on figure
    ax.set_xticklabels(labels=["neg","pos"], fontsize=13)
    ax.set_yticklabels(labels=["neg","pos"], fontsize= 13)
    plt.xlabel("\nactual value", fontsize=15)
    plt.ylabel("predicted value\n", fontsize=15)
    plt.show()
    
def plot_roc_curve(df):
    sns.lineplot(data=df, x='fpr', y='tpr', ci=None, estimator=None, sort=False)
    plt.fill_between(df['fpr'], df['tpr'], alpha=.5)
    # Add dashed line with a slope of 1
    plt.plot([0,1], [0,1], linestyle=(0, (5, 5)), linewidth=2, color='grey')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC curve");
    
def plot_pr_curve(df):
    sns.lineplot(data=df, x='recall', y='precision', ci=None, estimator=None, sort=False)
    plt.fill_between(df['recall'], df['precision'], alpha=.5)
    # Add dashed line with a slope of 1
    plt.plot([1,0], [0,1], linestyle=(0, (5, 5)), linewidth=2, color='grey')
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-recall curve");

In [ ]:
## show confustion matrix
show_confusion_matrix(y_test, y_pred)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
precision_score(y_test, y_pred)

In [ ]:
recall_score(y_test, y_pred)

In [ ]:
f1_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

### Receiver operating characteristic curve

Another common metric for classification problems is area under the receiver operating characteristic (ROC) curve. This plot, and associated scalar metric, assesses a model's performance by comparing the true positive rate (TPR) to the false positive rate (FPR) at varying thresholds. What this means is as the threshold varies from 0 to 1, can the model still successfully discern the two classes.

**Sensitivity/True positive rate (TPR):**
* TPR = TP / (TP + FN)
* TPR is the ability of a classifier to find all positive instances. 
* High TPR indicates a small number of false negatives.

**Specificity/True negative rate (TNR):**
* TNR = TN / (TN + FP)
* TNR is the ability of a classifier to find all negative instances. 
* High TNR indicates a small number of false positives.

**False positive rate (FPR):**
* FPR = FP / (FP + TN)
* FPR is the ability of a classifier to incorrectly predict positives instances. 
* Low FPR indications a small number of false positives.

<img width="300px" src="img/sensitivity-specificity.png" />


In [ ]:
# Calculate the area under the ROC curve
roc_auc_score(y_test, y_proba)

In [ ]:
# Calculate the FPR and TPR at each threshold
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# Tabulate the results in a dataframe
df_roc = pd.DataFrame(list(zip(fpr,tpr)), columns=['fpr','tpr'])
# Plot the curve
plot_roc_curve(df_roc)

### Precision recall curve

Related to the above ROC curve, the precision recall curve, and the area under it, is often used as a metric to determine classification model efficacy. This metric is especially important for imbalanced class sizes -- one class has far more samples (majority) than the other class (minority). The difference between this curve and the ROC curve is the use of precision instead of FPR (TPR and recall are the same metric). Precision is less affected by a large number of negative samples because it takes into account true positives and false positives. On the other hand, FPR is based upon false positives and true negatives so for a large number of negative samples this metric would very slowly increase. **Precision is focused on the positive class**

*Use AUPR when you have imbalanced classes, specifically when you have a majority of negative samples (usually the case).*

In [ ]:
# Calculate the precision and recall at each threshold
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
# Calculate the area under the PR curve
auc(recall, precision)

In [ ]:
# Tabulate the results in a dataframe
df_pr = pd.DataFrame(list(zip(recall,precision)), columns=['recall','precision'])
# Plot the curve
plot_pr_curve(df_pr)

<font color='red'>***Note***</font>: 
Interesting artifact of this plot, the rapid drop to the left of the plot is due to the way the curve is calculated. As you move right to left on the plot the threshold for determining if a prediction is postive becomes more stringent. So on the far left, there are very few positive class predictions (both true and false positives), so adding one more false or true positive (moving towards the right) can greatly affect the precision. 

### Interpretability and feature importance

We will analyze the model a bit more to understand what the model is doing and what features are most important to the classification.

First, we will extract the coefficients from the model and match them with the corresponding name of the features.

In [ ]:
coefs = list(zip(feature_cols,model.coef_[0]))
coefs

Next, we take the exponential of the coefficients to calculate the odds ratio for each feature.

In [ ]:
odds_ratio = [(x[0],math.exp(x[1])) for x in coefs]
odds_ratio

We see that of these features the most important appears to be Diabetes Pedigree Function. The odds ratio tells us that for every 1 unit increase in Diabetes Pedigree Function a patient is 2.53x more likely to experience the outcome (diabetes)

In [ ]:
[(x[0],(x[1]-1)*100.0) for x in odds_ratio]

## Changing the splitting

What happens if we change the training/testing split to 80/20...

In [ ]:
## split dataset into training set and test set
X2_train, X2_test, y2_train, y2_test = \
    train_test_split(X, y, test_size=0.2, random_state=42) # 80% training and 20% test

## create a logistic regression classifier and predict
model = LogisticRegression(random_state=42, max_iter=500)
model.fit(X2_train, y2_train)
y2_pred = model.predict(X2_test)
y2_proba = model.predict_proba(X2_test)[:,1]
print(f"Our model converged after {model.n_iter_[0]} iterations.")

print(classification_report(y2_test, y2_pred))

# Calculate the area under the ROC curve
print(f"AUROC = {roc_auc_score(y2_test, y2_proba)}\n")

# Calculate the precision and recall at each threshold
precision2, recall2, thresholds2 = precision_recall_curve(y2_test, y2_proba)
# Calculate the area under the PR curve
print(f"AUPR = {auc(recall2, precision2)}")

## Neural Networks

Now that we understand and have run a logistic regression model, let's go a bit "deeper". We can think of a neural network (NN) as a set of nested functions -- we call these layers. Each layer in our model takes input from the previous layer and outputs directly to the next layer, i.e. fully connected. 

We are going to create a 3 layer neural network with the previously used 8 variables as features and the "Outcome" as the label. 

The first layer of our NN will take in all 8 features as input, has a ReLU (rectified linear unit) activation function, and outputs 24 latent features (hidden). As opposed to the logistic function, discussed previously, ReLU sets the input to 0 if it is <0 or uses the input as is if >0.

$f(x)=max(0,x)$

The second layer of our NN will take in all 24 latent features from the previous layer as input, has a ReLU (rectified linear unit) activation function, and outputs 12 latent features.

The third (and last) layer of our model is a sigmoid output layer that takes in the previous 12 latent features as input.

The loss function we use for this model is binary cross entropy, which basically sums the log probabilty of a given sample being in the 0 class and the log probability of the sample being in the 1 class across all samples. This is essentially the same function as the log likelihood. We want to minimize this loss function.

$ \ln Loss = \sum_{i=1}^{N}-(y_{i}\ln f(x_{i})+(1-{y_{i}}) \ln (1-f(x_{i}))$


For our implementation of neural network, we will use keras's sequential model:
* https://keras.io/guides/sequential_model/

Let's load the libraries we will be using...

In [ ]:
import random
import numpy as np
import tensorflow as tf
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
from keras.models import Sequential
from keras.layers import Dense, Dropout

# summarize history for loss
def plot_fit(history):
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

In [ ]:
# define the keras model
model = Sequential()
model.add(Dense(24, input_dim=8, activation='relu'))
model.add(Dense(12, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# compile the keras model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['AUC'])
# fit the keras model on the dataset
history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, batch_size=10, verbose=False)

# make class predictions with the model
y_proba = model.predict(X_test)
y_pred = (y_proba > 0.5).astype("int32")

In [ ]:
plot_fit(history)

In [ ]:
## show confustion matrix
show_confusion_matrix(y_test, y_pred)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
# calculate F1 score
f1_score(y_test, y_pred)

In [ ]:
# Calculate AUROC
roc_auc_score(y_test, y_pred)

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# Tabulate the results in a dataframe
df_roc = pd.DataFrame(list(zip(fpr,tpr)), columns=['fpr','tpr'])
# Plot the ROC curve
plot_roc_curve(df_roc)

In [ ]:
# Calculate PR curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
# Calculate AUPR
auc(recall, precision)

In [ ]:
# Tabulate the results in a dataframe
df_pr = pd.DataFrame(list(zip(recall,precision)), columns=['recall','precision'])
# Plot the PR curve
plot_pr_curve(df_pr)

### Underfitting and Overfitting

The below figure is an excellent example of underfitting, a good fit, and overfitting.

<img width="800px" src="img/model_fit.png"/>

One can also plot learning curves to determine a good fit during training. This plots loss vs. epoch.

<img width="900px" src="img/learning_curves.png"/>

There are methods for avoiding overfitting/overtraining, such as regularization, dropout, etc. You can learn more about these in many of the references provided.

https://scikit-learn.org/stable/model_selection.html

*Choosing the right model(s), activation function(s), and **hyperparameters** are crucial for creating a robust and **generalizable** model.*

## Discussion topics
 
* Interpretability of different models
* Handling high dimensionality of biomedical data
* Models that account for longitudinal data for patient populations
* Missingness in datasets -- imputation and omission
* Generalizability of models for biomedical applications



## References and additional reading

In this module, we covered the basics of implementing and evaluating a logistic regression classifier in scikit learns and a neural network using keras. 

* Burkov A. The Hundred-Page Machine Learning Book by Andriy Burkov. Expert Systems. 2019;5(2):132-50.
* Pedregosa F, Varoquaux G, Gramfort A, Michel V, Thirion B, Grisel O, Blondel M, Prettenhofer P, Weiss R, Dubourg V, Vanderplas J. Scikit-learn: Machine learning in Python. the Journal of machine Learning research. 2011 Nov 1;12:2825-30.
* Chollet F. Keras documentation. keras.io. 2015;33.
* Goodfellow I, Bengio Y, Courville A. Deep learning.
* Bishop C. Pattern Recognition and Machine Learning.
* Friedman JH, Tibshirani R, Hastie T. The Elements of Statistical Learning.
* https://www.codecademy.com/learn/machine-learning
* https://www.w3schools.com/python/python_ml_getting_started.asp
* https://machinelearningmastery.com/learning-curves-for-diagnosing-machine-learning-model-performance/